In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, roc_auc_score,
                              roc_curve)
from sklearn.preprocessing import StandardScaler
df = pd.read_csv('heart_disease_uci.csv')
print(df.shape)
df.head()



In [ ]:
# Convert multiclass target to binary
df['num'] = (df['num'] > 0).astype(int)
# Check for missing values
print('Missing values:')
print(df.isnull().sum())
# Check target column distribution
print('Target distribution:')
print(df['num'].value_counts())
# 1 = has heart disease, 0 = no heart disease

In [ ]:
# Distribution of ages
plt.figure(figsize=(8,4))
sns.histplot(df['age'], bins=20, kde=True, color='steelblue')
plt.title('Age Distribution of Patients')
plt.xlabel('Age'); plt.ylabel('Count')
plt.tight_layout(); plt.show()

# Correlation heatmap
# First, convert categorical columns to numerical using one-hot encoding for correlation calculation
df_encoded = pd.get_dummies(df.copy(), columns=['sex', 'dataset', 'cp', 'restecg', 'exang', 'slope', 'thal'], drop_first=True)

# Handle 'ca' and 'fbs' that might still be objects if they contain non-numeric string values (like '?' in some datasets)
# and convert them to numeric, coercing errors to NaN
# Note: This is an assumption based on common issues in such datasets. Reviewing df.info() or df.head() would confirm.
for col in ['ca', 'fbs', 'trestbps', 'chol', 'thalch', 'oldpeak']:
    if col in df_encoded.columns:
        df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')

# Drop columns that are completely NaN after coercion or are not relevant for correlation (e.g., 'id')
df_encoded = df_encoded.dropna(axis=1, how='all')
df_encoded = df_encoded.drop(columns=['id'], errors='ignore')

plt.figure(figsize=(12, 10))
sns.heatmap(df_encoded.corr(), annot=False, fmt='.2f', cmap='coolwarm') # annot=False due to potentially many columns
plt.title('Feature Correlation Heatmap (with one-hot encoded categoricals)')
plt.tight_layout(); plt.show()

In [ ]:
# Drop the 'num' target column to create X (features)
X = df.drop('num', axis=1)
y = df['num']                # what we want to predict

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include='object').columns

# Apply one-hot encoding to X
X_encoded = pd.get_dummies(X.copy(), columns=categorical_cols, drop_first=True)

# Handle columns that might still be objects due to non-numeric string values (like '?' in some datasets)
# and convert them to numeric, coercing errors to NaN. Assuming these columns are present in X_encoded.
for col in ['ca', 'fbs', 'trestbps', 'chol', 'thalch', 'oldpeak']:
    if col in X_encoded.columns:
        X_encoded[col] = pd.to_numeric(X_encoded[col], errors='coerce')

# Drop any columns that became entirely NaN after coercion (if they contained only '?' or similar)
X_encoded = X_encoded.dropna(axis=1, how='all')

# Drop columns that are not relevant for training (e.g., 'id')
X_encoded = X_encoded.drop(columns=['id'], errors='ignore')

# Split: 80% train, 20% test using the processed X_encoded
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
from sklearn.impute import SimpleImputer

# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)

# Impute missing values (NaNs) in X_train and X_test
imputer = SimpleImputer(strategy='mean') # You can choose 'median', 'most_frequent', etc.

# Fit on X_train and transform both X_train and X_test
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

model.fit(X_train_imputed, y_train)

# Make predictions
y_pred = model.predict(X_test_imputed)
y_prob = model.predict_proba(X_test_imputed)[:, 1]  # probability scores
# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc*100:.2f}%')
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease','Disease'],
            yticklabels=['No Disease','Disease'])
plt.title('Confusion Matrix')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0,1],[0,1],'k--',lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Heart Disease Prediction')
plt.legend(); plt.tight_layout(); plt.show()

## Task 3: Heart Disease Prediction — Summary

### Objective
To build a classification model that predicts whether
a patient is at risk of heart disease using health data.

### Dataset
- **Name:** Heart Disease UCI Dataset
- **File:** heart_disease_uci.csv
- **Size:** 920 patients, 16 columns
- **Note:** Used full UCI dataset

### Binary Classification
- **0** = No Heart Disease (411 patients)
- **1** = Has Heart Disease (509 patients)

### Data Preprocessing
- Converted multiclass target (0-4) to binary (0 or 1)
- Detected and handled missing values using SimpleImputer
- Applied One-Hot Encoding on categorical columns
- Applied StandardScaler for feature scaling

### Model Used
- **Algorithm:** Logistic Regression
- **Train/Test Split:** 80% training, 20% testing

### Results
  Accuracy:  83 percent


### Visualizations
- Age Distribution Histogram
- Feature Correlation Heatmap
- Confusion Matrix
- ROC Curve

### Key Finding
The model successfully classifies heart disease risk
from patient health data using Logistic Regression.


